# Filter rows in CSV where EX column is 0 and save to new CSV

In [2]:
# Read CSV and filter rows where EX column is 0
import pandas as pd

input_csv_path = "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/new_sql_queries.csv"
output_csv_path = "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/new_sql_nl_queries_alter_failed.csv"

# Read the CSV file
df = pd.read_csv(input_csv_path)

print(f"Total rows in original CSV: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Filter rows where EX column is 0
df_filtered = df[df['EX'] == 0]

print(f"Rows with EX = 0: {len(df_filtered)}")

# Write to new CSV file
df_filtered.to_csv(output_csv_path, index=False)

print(f"\nFiltered data saved to: {output_csv_path}")


Total rows in original CSV: 1534
Columns: ['question_id', 'db_id', 'question', 'evidence', 'original_sql', 'new_sql', 'difficulty', 'EX']
Rows with EX = 0: 1

Filtered data saved to: /Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/new_sql_nl_queries_alter_failed.csv


# Check example of altered SQL failure

In [3]:
import sys

from src.core.database.database_handler import DatabaseHandler, DBMS
from experiments.textual_query_augmentation.schema_linking.schema_linker import SchemaReader
from src.metrics.evaluation import Evaluation, EvaluationTechnique
from src.core.model_manager import OpenAIModel
import pandas as pd

# Database configurations
original_config = {
    "dbms": "sqlite",
    "db_dir_path": "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/benchmarks/Bird/dev_databases/",
    "tables_file_path": "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/benchmarks/Bird/dev_tables.json",
}

new_config = {
    "dbms": "sqlite",
    "db_dir_path": "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/new_dev_databases",
    "tables_file_path": "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/new_dev_databases/new_dev_tables.json",
}

# SQL queries
db_name = "codebase_community"
gold_sql = "SELECT CAST(SUM(CASE WHEN T2.Score > 50 THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(T1.Id) FROM users T1 INNER JOIN posts T2 ON T1.Id = T2.OwnerUserId INNER JOIN ( SELECT MAX(Reputation) AS max_reputation FROM users ) T3 ON T1.Reputation = T3.max_reputation"
new_sql = "SELECT CAST(SUM(CASE WHEN T2.Scr > 50 THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(T1.Id) FROM Usr T1 INNER JOIN pts T2 ON T1.Id = T2.OwnUsrId INNER JOIN ( SELECT MAX(Rptn) AS max_reputation FROM Usr ) T3 ON T1.Rptn = T3.max_reputation"
# Construct database paths
original_db_path = f"{original_config['db_dir_path']}/{db_name}/{db_name}.sqlite"
new_db_path = f"{new_config['db_dir_path']}/{db_name}/{db_name}.sqlite"

# Get schemas using SchemaReader
print("="*60)
print("ORIGINAL DATABASE SCHEMA")
print("="*60)
original_schema_reader = SchemaReader(
    database_name=db_name,
    tables_file_path=original_config["tables_file_path"],
    dbs_dir_path=original_config["db_dir_path"]
)
original_schema, original_columns = original_schema_reader.get_full_schema_representation()
# print(original_schema)
print("\nColumns by table:")
print(original_columns)

print("\n" + "="*60)
print("NEW DATABASE SCHEMA")
print("="*60)
new_schema_reader = SchemaReader(
    database_name=db_name,
    tables_file_path=new_config["tables_file_path"],
    dbs_dir_path=new_config["db_dir_path"]
)
new_schema, new_columns = new_schema_reader.get_full_schema_representation()
# print(new_schema)
print("\nColumns by table:")
print(new_columns)

# Create database handlers
original_handler = DatabaseHandler(
    dbms=DBMS.SQLITE,
    connection_params={"db_path": original_db_path}
)

new_handler = DatabaseHandler(
    dbms=DBMS.SQLITE,
    connection_params={"db_path": new_db_path}
)

# Run queries
print("\n" + "="*60)
print("QUERY EXECUTION")
print("="*60)
print("\nRunning Gold SQL on ORIGINAL database:")
print(f"SQL: {gold_sql}")
original_results = original_handler.run_query(gold_sql)
print("HREe")
print(f"Results: {original_results}\n")

print("Running New SQL on NEW database:")
print(f"SQL: {new_sql}")
new_results = new_handler.run_query(new_sql)
print(f"Results: {new_results}")

# Calculate EX metric - Compare gold_sql results on original DB vs new_sql results on new DB
print("\n" + "="*60)
print("EXECUTION ACCURACY (EX) METRIC")
print("="*60)

# Execute queries and calculate EX using the same logic as execution_accuracy.py
new_db_handler = None
original_db_handler = None
has_error = False
error_message = None

try:
    # Execute new_sql on new database
    new_db_handler = DatabaseHandler(
        dbms=DBMS.SQLITE,
        connection_params={"db_path": new_db_path}
    )
    pred_cols, pred_rows = new_db_handler.run_query(new_sql)

    # Execute gold_sql on original database
    original_db_handler = DatabaseHandler(
        dbms=DBMS.SQLITE,
        connection_params={"db_path": original_db_path}
    )
    gt_cols, gt_rows = original_db_handler.run_query(gold_sql)

    # Calculate binary execution accuracy (same logic as match_columns in execution_accuracy.py)
    ex = 1 if set(gt_rows) == set(pred_rows) else 0

    print(f"\nGold SQL on Original DB Results: {gt_rows}")
    print(f"New SQL on New DB Results: {pred_rows}")

    if ex == 1:
        print(f"\n✓ Results MATCH - EX Score: {ex}")
    else:
        print(f"\n✗ Results DO NOT MATCH - EX Score: {ex}")

except Exception as e:
    has_error = True
    error_message = str(e)
    ex = 0
    print(f"\nError occurred: {error_message}")
    print(f"EX Score: {ex}")

finally:
    if new_db_handler:
        new_db_handler.close_connection()
    if original_db_handler:
        original_db_handler.close_connection()

# Close connections
original_handler.close_connection()
new_handler.close_connection()


ORIGINAL DATABASE SCHEMA

Columns by table:
{'badges': ['Id', 'UserId', 'Name', 'Date'], 'comments': ['Id', 'PostId', 'Score', 'Text', 'CreationDate', 'UserId', 'UserDisplayName'], 'postHistory': ['Id', 'PostHistoryTypeId', 'PostId', 'RevisionGUID', 'CreationDate', 'UserId', 'Text', 'Comment', 'UserDisplayName'], 'postLinks': ['Id', 'CreationDate', 'PostId', 'RelatedPostId', 'LinkTypeId'], 'posts': ['Id', 'PostTypeId', 'AcceptedAnswerId', 'CreaionDate', 'Score', 'ViewCount', 'Body', 'OwnerUserId', 'LasActivityDate', 'Title', 'Tags', 'AnswerCount', 'CommentCount', 'FavoriteCount', 'LastEditorUserId', 'LastEditDate', 'CommunityOwnedDate', 'ParentId', 'ClosedDate', 'OwnerDisplayName', 'LastEditorDisplayName'], 'tags': ['Id', 'TagName', 'Count', 'ExcerptPostId', 'WikiPostId'], 'users': ['Id', 'Reputation', 'CreationDate', 'DisplayName', 'LastAccessDate', 'WebsiteUrl', 'Location', 'AboutMe', 'Views', 'UpVotes', 'DownVotes', 'AccountId', 'Age', 'ProfileImageUrl'], 'votes': ['Id', 'PostId', '

# Check for duplicated table and column names within each database


In [ ]:
# Check for duplicated table and column names within each database
import pandas as pd
from collections import Counter

csv_path = "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/databases_naturalness_decreased_fixed.csv"

# Read the CSV file
df = pd.read_csv(csv_path)

print(f"Total rows in CSV: {len(df)}")
print(f"Columns: {df.columns.tolist()}\n")

# Group by dataset and db_id to check each database separately
grouped = df.groupby(['dataset', 'db_id'])

duplicates_found = False

for (dataset, db_id), group in grouped:
    print("="*80)
    print(f"Database: {dataset} / {db_id}")
    print("="*80)

    # Check for duplicate new table names
    new_table_names = group['new_table_name'].dropna().tolist()
    table_counts = Counter(new_table_names)
    duplicate_tables = {name: count for name, count in table_counts.items() if count > 1}

    if duplicate_tables:
        duplicates_found = True
        print(f"\n⚠️  DUPLICATE NEW TABLE NAMES FOUND:")
        for table, count in duplicate_tables.items():
            print(f"   - '{table}' appears {count} times")
            # Show original table names that map to this duplicate
            original_tables = group[group['new_table_name'] == table]['table_name'].unique()
            print(f"     Original tables: {list(original_tables)}")
    else:
        print(f"\n✓ No duplicate new table names")

    # Check for duplicate new column names within the same table
    table_column_duplicates = []
    for table_name in group['table_name'].unique():
        table_group = group[group['table_name'] == table_name]
        new_column_names = table_group['new_column_name'].dropna().tolist()
        column_counts = Counter(new_column_names)
        duplicate_columns = {name: count for name, count in column_counts.items() if count > 1}

        if duplicate_columns:
            table_column_duplicates.append((table_name, duplicate_columns))

    if table_column_duplicates:
        duplicates_found = True
        print(f"\n⚠️  DUPLICATE NEW COLUMN NAMES FOUND:")
        for table_name, duplicate_columns in table_column_duplicates:
            new_table_name = group[group['table_name'] == table_name]['new_table_name'].iloc[0]
            print(f"   Table: {table_name} (new: {new_table_name})")
            for column, count in duplicate_columns.items():
                print(f"      - '{column}' appears {count} times")
                # Show original column names
                original_columns = group[(group['table_name'] == table_name) &
                                        (group['new_column_name'] == column)]['column_name'].tolist()
                print(f"        Original columns: {original_columns}")
    else:
        print(f"✓ No duplicate new column names within tables")

    print()

if not duplicates_found:
    print("\n" + "="*80)
    print("✓ NO DUPLICATES FOUND ACROSS ALL DATABASES")
    print("="*80)
else:
    print("\n" + "="*80)
    print("⚠️  DUPLICATES DETECTED - REVIEW REQUIRED")
    print("="*80)


# Check table structure using PRAGMA table_info


In [37]:
# Check the actual table structure in the new database using PRAGMA
from src.core.database.database_handler import DatabaseHandler, DBMS

db_name = "thrombosis_prediction"
new_db_path = f"/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/new_dev_databases/{db_name}/{db_name}.sqlite"

# Create database handler for new database
new_handler = DatabaseHandler(
    dbms=DBMS.SQLITE,
    connection_params={"db_path": new_db_path}
)

print("="*60)
print("PRAGMA table_info(acct) - New Database")
print("="*60)

# Run PRAGMA query to get table structure
pragma_query = "PRAGMA table_info(Exmtn);"
cols, rows = new_handler.run_query(pragma_query)

print(f"\nColumns: {cols}")
print(f"\nTable structure:")
for row in rows:
    print(f"  {row}")

# Close connection
new_handler.close_connection()


2026-01-20 22:22:26 | INFO     | sqlite_adapter:connect:24 | {"action": "CONNECTION_ESTABLISHED", "db_path": "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/new_dev_databases/thrombosis_prediction/thrombosis_prediction.sqlite"}
PRAGMA table_info(acct) - New Database
2026-01-20 22:22:26 | INFO     | sqlite_adapter:run_query:34 | {"action": "QUERY_EXECUTED", "query": "PRAGMA table_info(Exmtn);"}

Columns: ['cid', 'name', 'type', 'notnull', 'dflt_value', 'pk']

Table structure:
  (0, 'ID', 'INTEGER', 0, None, 0)
  (1, 'ExmtnDt', 'DATE', 0, None, 0)
  (2, 'aCL IgG', 'REAL', 0, None, 0)
  (3, 'aCL IgM', 'REAL', 0, None, 0)
  (4, 'ANA', 'INTEGER', 0, None, 0)
  (5, 'ANAPtn', 'TEXT', 0, None, 0)
  (6, 'aCL IgA', 'INTEGER', 0, None, 0)
  (7, 'Dx', 'TEXT', 0, None, 0)
  (8, 'KCT', 'TEXT', 0, None, 0)
  (9, 'RVVT', 'TEXT', 0, None, 0)
  (10, 'LAC', 'TEXT', 0, None, 0)
  (11, 'Symp', 'TEXT', 0, None, 0)
  (12, 'Thrmbs', 'IN

# Filter CSV by question_id from failed queries


In [7]:
import pandas as pd

# Read the CSV with failed question_ids
failed_csv_path = "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/experiments_dev_new_sql_nl_queries_failed.csv"
df_failed = pd.read_csv(failed_csv_path)

# Extract question_ids
failed_question_ids = df_failed['question_id'].tolist()
print(f"Number of failed question_ids: {len(failed_question_ids)}")
print(f"Sample question_ids: {failed_question_ids[:5]}")

# Read the complete CSV file
complete_csv_path = "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/new_sql_queries.csv"
df_complete = pd.read_csv(complete_csv_path)
print(f"\nTotal rows in complete CSV: {len(df_complete)}")
print(f"Columns in complete CSV: {df_complete.columns.tolist()}")

# Filter complete CSV to only keep rows with matching question_ids
df_complete_filtered = df_complete[df_complete['question_id'].isin(failed_question_ids)]
print(f"Filtered rows matching failed question_ids: {len(df_complete_filtered)}")

# Select specific columns from each dataframe
# From failed CSV: question_id, db_id, original_question, original_evidence, new_question, new_evidence, difficulty
# From complete CSV: original_sql, new_sql, EX
df_from_failed = df_failed[['question_id', 'db_id', 'original_question', 'original_evidence',
                              'new_question', 'new_evidence', 'difficulty']]

df_from_complete = df_complete_filtered[['question_id', 'original_sql', 'new_sql', 'EX']]

# Merge on question_id
df_output = df_from_failed.merge(df_from_complete, on='question_id', how='left')

# Reorder columns to match the exact order specified
df_output = df_output[['question_id', 'db_id', 'original_question', 'original_evidence',
                        'new_question', 'new_evidence', 'original_sql', 'new_sql', 'difficulty', 'EX']]

print(f"\nFinal output rows: {len(df_output)}")
print(f"Columns in output: {df_output.columns.tolist()}")

# Save to new CSV
output_csv_path = "/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/experiments_dev_new_sql_nl_queries_fixed.csv"
df_output.to_csv(output_csv_path, index=False)

print(f"\nFiltered data saved to: {output_csv_path}")


Number of failed question_ids: 333
Sample question_ids: [1, 2, 26, 30, 37]

Total rows in complete CSV: 1534
Columns in complete CSV: ['question_id', 'db_id', 'question', 'evidence', 'original_sql', 'new_sql', 'difficulty', 'EX']
Filtered rows matching failed question_ids: 333

Final output rows: 333
Columns in output: ['question_id', 'db_id', 'original_question', 'original_evidence', 'new_question', 'new_evidence', 'original_sql', 'new_sql', 'difficulty', 'EX']

Filtered data saved to: /Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/augmentation/decrease_naturalness/experiment_dev_sql_failed/experiments_dev_new_sql_nl_queries_fixed.csv
